# Project: Forecasting Approach Evaluation and Recommendation

The board meeting is in two weeks. Your VP of Sales presented a revenue target of **$4.1M per month by December 2026** — 18% year-over-year growth. The number came from pasting a data export into a generative AI tool and asking it to forecast.

Your data engineering team has already audited the VP's original query and found several bugs that inflated recent revenue figures and made the growth trend look steeper than it actually is. They've corrected the issues and handed you a **clean dataset** (`revenue.csv`) with 60 months of monthly revenue from January 2021 through December 2025.

**Your job:** Take the clean data and figure out what the numbers actually say. Build progressively sophisticated forecasts — from simple baselines through classical ARIMA/SARIMAX to neural models and foundation models — and deliver a stakeholder-ready recommendation with prediction intervals that a non-technical executive can act on.

---

## Deliverables

1. This completed notebook with all phases implemented
2. A comparison table showing all models, their December 2026 forecasts, prediction intervals, and accuracy metrics
3. At least three charts: baseline comparison, classical model forecasts with intervals, modern model forecasts
4. A written recommendation (500–1000 words) answering:
   - What is the realistic range for December 2026 revenue?
   - What should the company plan for instead of $4.1M?
   - Which forecasting approach would you recommend for ongoing use, and why?

## Setup

In [ ]:
!pip install scipy statsmodels darts pytorch_lightning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from darts import TimeSeries
from darts.models import NBEATSModel, ARIMA as DartsARIMA
from darts.metrics import mae, rmse, mape
from darts.utils.likelihood_models import QuantileRegression
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

VP_TARGET = 4_100_000
HORIZON = 12  # months ahead (Jan 2026 through Dec 2026)

## Load Data

The clean revenue dataset has been prepared by your data engineering team. It contains monthly revenue from January 2021 through December 2025.

In [ ]:
df = pd.read_csv(
    "../data/revenue.csv",
    parse_dates=["date"],
    index_col="date",
)
df.index.freq = "MS"
revenue = df["revenue"]

print(f"Date range: {revenue.index[0].date()} to {revenue.index[-1].date()}")
print(f"Number of months: {len(revenue)}")
print(f"Revenue range: ${revenue.min()/1e6:.2f}M to ${revenue.max()/1e6:.2f}M")

revenue.plot(figsize=(14, 5), color="black", title="Monthly Revenue")
plt.ylabel("Revenue ($)")
plt.tight_layout()
plt.show()

## Phase 1: Baseline Forecasts

Before reaching for sophisticated models, establish baselines. Compute simple forecasts over a 12-month horizon and compare each to the VP's $4.1M target.

### Helper Functions

```python
def naive_forecast(series, horizon):
    """Repeat the last observed value for `horizon` months."""
    last = series.iloc[-1]
    future = pd.date_range(
        series.index[-1] + pd.DateOffset(months=1), periods=horizon, freq="MS"
    )
    return pd.Series(last, index=future)


def moving_average_forecast(series, window, horizon):
    """Repeat the trailing `window`-month average for `horizon` months."""
    avg = series.iloc[-window:].mean()
    future = pd.date_range(
        series.index[-1] + pd.DateOffset(months=1), periods=horizon, freq="MS"
    )
    return pd.Series(avg, index=future)


def trend_forecast(series, horizon):
    """Fit a linear trend and extrapolate for `horizon` months."""
    t = np.arange(len(series))
    coeffs = np.polyfit(t, series.values, 1)
    future_t = np.arange(len(series), len(series) + horizon)
    future = pd.date_range(
        series.index[-1] + pd.DateOffset(months=1), periods=horizon, freq="MS"
    )
    return pd.Series(np.polyval(coeffs, future_t), index=future)


def plot_forecasts(actuals, forecasts, target=VP_TARGET, title="Forecast Comparison"):
    """Plot actuals and multiple forecast series against the VP target."""
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(actuals.index, actuals / 1e6, color="black", linewidth=1.5, label="Actual")
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:purple", "tab:red"]
    for i, (name, fc) in enumerate(forecasts.items()):
        ax.plot(fc.index, fc / 1e6, "--", color=colors[i % len(colors)], label=name)
    ax.axhline(
        y=target / 1e6, color="red", linestyle=":", linewidth=2,
        label=f"VP target: ${target/1e6:.1f}M",
    )
    ax.set_ylabel("Revenue ($M)")
    ax.set_title(title)
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()
    return fig
```

### TODO: Generate Baseline Forecasts

Create naive, 6-month MA, 12-month MA, and linear trend forecasts. Compare each December 2026 forecast to the VP target.

In [ ]:
def naive_forecast(series, horizon):
    """Repeat the last observed value for `horizon` months."""
    last = series.iloc[-1]
    future = pd.date_range(
        start=series.index[-1] + pd.DateOffset(months=...),
        periods=horizon,
        freq="MS"
    )
    return pd.Series([last] * horizon, index=future)


def moving_average_forecast(series, window, horizon):
    """Repeat the trailing `window`-month average for `horizon` months."""
    avg = series.iloc[...:...].mean()
    future = pd.date_range(
        series.index[-1] + pd.DateOffset(months=1),
        periods=horizon,
        freq="MS"
    )
    return pd.Series([avg] * horizon, index=future)


def trend_forecast(series, horizon):
    """Fit a linear trend and extrapolate for `horizon` months."""
    t = np.arange(len(series))
    coeffs = np.polyfit(t, series.values, 1)
    future_t = np.arange(len(series), len(series) + horizon)
    future = pd.date_range(
        series.index[-1] + pd.DateOffset(months=1),
        periods=horizon,
        freq="MS"
    )
    return pd.Series(np.polyval(coeffs, ...), index=future)


def plot_forecasts(actuals, forecasts, target=VP_TARGET, title="Forecast Comparison"):
    """Plot actuals and multiple forecast series against the VP target."""
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(actuals.index, actuals / 1e6, color="black", linewidth=1.5, label="Actual")
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:purple", "tab:red"]
    for i, (name, fc) in enumerate(forecasts.items()):
        ax.plot(fc.index, fc / 1e6, "--", color=colors[i % len(colors)], label=name)
    ax.axhline(
        y=target / 1e6, color="red", linestyle=":", linewidth=2,
        label=f"VP target: ${target/1e6:.1f}M",
    )
    ax.set_ylabel("Revenue ($M)")
    ax.set_title(title)
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
# PHASE 1 TODO: Select baseline methods and generate forecasts
#
# Consider: Which baseline methods are most appropriate for this data?
# - Naive: assumes no trend or seasonality
# - Moving average: smooths out short-term fluctuations  
# - Trend: assumes linear growth continues
#
# TODO: Decide which window size(s) to try for moving average
# (e.g., 3 months captures recent trends, 12 months captures annual patterns)

# Generate forecasts using chosen methods
forecasts = {
    'Naive': naive_forecast(revenue, horizon=...),
    '6MA': moving_average_forecast(revenue, window=..., horizon=...),
    '12MA': moving_average_forecast(revenue, window=..., horizon=...),
    'Trend': trend_forecast(revenue, horizon=...)
}

# Compare December 2026 forecasts to VP target
for name, fc in forecasts.items():
    dec_2026 = fc.iloc[-1]
    gap = dec_2026 - VP_TARGET
    print(f"{name}: ${dec_2026/1e6:.2f}M (gap: ${gap/1e6:+.2f}M)")

In [ ]:
plot_forecasts(revenue, forecasts, VP_TARGET, "Phase 1: Baseline Forecasts")

---

## Phase 2: Classical Models

Fit ARIMA and SARIMAX models. Experiment with different parameter specifications to see which ones converge and produce reasonable forecasts.

### Train/Test Split

Split at December 2024. Train on data through December 2024, test on January 2025 through December 2025.

In [ ]:
train = revenue[:'2024-12']
test = revenue['2025-01':]

print(f"Train: {train.index[0]} to {train.index[-1]} ({len(train)} months)")
print(f"Test:  {test.index[0]} to {test.index[-1]} ({len(test)} months)")

---

## Phase 2: Classical Models

Fit ARIMA and SARIMAX models. Experiment with different parameter specifications to see which ones converge and produce reasonable forecasts.

### Train/Test Split

Split at December 2024. Train on data through December 2024, test on January 2025 through December 2025.

In [ ]:
# Start with ARIMA(2,1,1) with trend='c'
arima_211 = SARIMAX(train, order=(2, 1, 1), trend='c').fit(disp=False)
print(f"ARIMA(2,1,1) AIC: {arima_211.aic:.2f}")

In [ ]:
# Try an alternative ARIMA specification
# TODO: Choose your own (p,d,q) order and trend setting
# Hint: Try fewer parameters if this one doesn't converge, or drop the trend
arima_alt = SARIMAX(train, order=(..., ..., ...), trend=...).fit(disp=False)
print(f"Alternative ARIMA AIC: {arima_alt.aic:.2f}")

In [ ]:
# Try a third ARIMA specification
# TODO: Choose different (p,d,q) or seasonal parameters
arima_third = SARIMAX(train, order=(..., ..., ...), trend=...).fit(disp=False)
print(f"Third ARIMA AIC: {arima_third.aic:.2f}")

In [ ]:
# Compare AICs across your explored models
# TODO: Add print statements for each model you tried
print("AIC Comparison:")
print(f"  ARIMA(2,1,1): {arima_211.aic:.2f}")
# Add your alternative models here

# Which has the lowest AIC?

In [ ]:
# Select best ARIMA based on exploration
best_arima = arima_211  # or arima_alt, arima_third

# Generate forecasts for test period
arima_forecast = best_arima.get_forecast(steps=...)
arima_mean = arima_forecast.predicted_mean

arima_mae = np.mean(np.abs(...))
arima_rmse = np.sqrt(np.mean(np.square(...)))

print(f"Best ARIMA Test MAE: ${arima_mae:,.0f}")
print(f"Best ARIMA Test RMSE: ${arima_rmse:,.0f}")

### Phase 2b: SARIMAX Parameter Exploration

Try different SARIMAX specifications with seasonal components.

**Start with:** SARIMAX(1,1,1)(1,1,1,12)

In [ ]:
# Start with SARIMAX(1,1,1)(1,1,1,12)
sarimax_1 = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12)
).fit(disp=False)
print(f"SARIMAX(1,1,1)(1,1,1,12) AIC: {sarimax_1.aic:.2f}")

In [ ]:
# Try alternative SARIMAX specification
# TODO: Choose your own order and seasonal_order
sarimax_2 = SARIMAX(
    train,
    order=(..., ..., ...),
    seasonal_order=(..., ..., ..., ...)
).fit(disp=False)
print(f"Alternative SARIMAX AIC: {sarimax_2.aic:.2f}")

In [ ]:
# Compare AICs across your explored models
# TODO: Add print statements for each model you tried
print("SARIMAX AIC Comparison:")
print(f"  Spec 1: {sarimax_1.aic:.2f}")
# Add your alternative models here

In [ ]:
# Select best SARIMAX based on exploration
best_sarimax = sarimax_1  # or sarimax_2

sarimax_forecast = best_sarimax.get_forecast(steps=...)
sarimax_mean = sarimax_forecast.predicted_mean

sarimax_mae = np.mean(np.abs(...))
sarimax_rmse = np.sqrt(np.mean(np.square(...)))

print(f"Best SARIMAX Test MAE: ${sarimax_mae:,.0f}")
print(f"Best SARIMAX Test RMSE: ${sarimax_rmse:,.0f}")

### Phase 2c: Residual Diagnostics

In [ ]:
# ARIMA diagnostics
print(f"ARIMA AIC: {best_arima.aic:.2f}")

lb_results = acorr_ljungbox(best_arima.resid, lags=10)
print(f"Ljung-Box p-value: {lb_results['lb_pvalue'].iloc[-1]:.4f}")

jb_stat, jb_pvalue = stats.jarque_bera(best_arima.resid)
print(f"Jarque-Bera p-value: {jb_pvalue:.4f}")

In [ ]:
# SARIMAX diagnostics
print(f"SARIMAX AIC: {best_sarimax.aic:.2f}")

lb_results = acorr_ljungbox(best_sarimax.resid, lags=10)
print(f"Ljung-Box p-value: {lb_results['lb_pvalue'].iloc[-1]:.4f}")

jb_stat, jb_pvalue = stats.jarque_bera(best_sarimax.resid)
print(f"Jarque-Bera p-value: {jb_pvalue:.4f}")

### Phase 2d: Refit on Full Data and Forecast to December 2026

In [ ]:
# Refit best ARIMA on full dataset
final_arima = SARIMAX(
    revenue,
    order=(..., ..., ...),  # Your chosen order
    trend=...              # Your chosen trend
).fit(disp=False)

arima_fc = final_arima.get_forecast(steps=HORIZON)
arima_mean = arima_fc.predicted_mean
arima_ci = arima_fc.conf_int(alpha=0.05)

fc_index = pd.date_range('2026-01-01', periods=HORIZON, freq='MS')
arima_mean.index = fc_index
arima_ci.index = fc_index

arima_dec_2026 = arima_mean.iloc[-1]
arima_lower = arima_ci.iloc[-1, 0]
arima_upper = arima_ci.iloc[-1, 1]

print(f"ARIMA Dec 2026: ${arima_dec_2026/1e6:.2f}M")
print(f"95% CI: [${arima_lower/1e6:.2f}M, ${arima_upper/1e6:.2f}M]")

In [ ]:
# Refit best SARIMAX
final_sarimax = SARIMAX(
    revenue,
    order=(..., ..., ...),
    seasonal_order=(..., ..., ..., ...),
    trend=...
).fit(disp=False)

sarimax_fc = final_sarimax.get_forecast(steps=HORIZON)
sarimax_mean = sarimax_fc.predicted_mean
sarimax_ci = sarimax_fc.conf_int(alpha=0.05)

sarimax_mean.index = fc_index
sarimax_ci.index = fc_index

sarimax_dec_2026 = sarimax_mean.iloc[-1]
sarimax_lower = sarimax_ci.iloc[-1, 0]
sarimax_upper = sarimax_ci.iloc[-1, 1]

print(f"SARIMAX Dec 2026: ${sarimax_dec_2026/1e6:.2f}M")
print(f"95% CI: [${sarimax_lower/1e6:.2f}M, ${sarimax_upper/1e6:.2f}M]")

In [ ]:
# Calculate probability of reaching $4.1M

def prob_above_target(mean, lower_ci, upper_ci, target):
    se = (upper_ci - mean) / 1.96
    z = (target - mean) / se
    return 1 - stats.norm.cdf(z)

arima_prob = prob_above_target(arima_dec_2026, arima_lower, arima_upper, VP_TARGET)
sarimax_prob = prob_above_target(sarimax_dec_2026, sarimax_lower, sarimax_upper, VP_TARGET)

print(f"P(reach $4.1M):")
print(f"  ARIMA:   {arima_prob:.1%}")
print(f"  SARIMAX: {sarimax_prob:.1%}")

In [ ]:
# Plot with prediction intervals
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(revenue.index, revenue / 1e6, 'k-', label='Actual')

ax.plot(fc_index, arima_mean / 1e6, 'b--', label='ARIMA')
ax.fill_between(fc_index, arima_ci.iloc[:, 0] / 1e6, arima_ci.iloc[:, 1] / 1e6, 
                color='blue', alpha=0.2)

ax.plot(fc_index, sarimax_mean / 1e6, 'orange', linestyle='--', label='SARIMAX')
ax.fill_between(fc_index, sarimax_ci.iloc[:, 0] / 1e6, sarimax_ci.iloc[:, 1] / 1e6,
                color='orange', alpha=0.2)

ax.axhline(y=VP_TARGET / 1e6, color='r', linestyle=':', label='VP Target: $4.1M')

ax.set_ylabel('Revenue ($M)')
ax.set_title('Phase 2: Classical Model Forecasts')
ax.legend()
plt.tight_layout()
plt.show()

---

## Phase 3: Modern Models

Train N-BEATS and run Chronos-2 using Darts.

In [ ]:
ts = TimeSeries.from_dataframe(
    df.reset_index(), 
    time_col='date', 
    value_cols='revenue', 
    freq='MS'
)

train_ts, test_ts = ts.split_before(pd.Timestamp('2025-01-01'))

print(f"Train: {len(train_ts)} months")
print(f"Test: {len(test_ts)} months")

In [ ]:
nbeats = NBEATSModel(
    input_chunk_size=...,
    output_chunk_size=...,
    n_epochs=...,
    verbose=True
)

nbeats.fit(train_ts)

nbeats_pred = nbeats.predict(len(test_ts))
nbeats_mae = mae(test_ts, nbeats_pred)
nbeats_rmse = rmse(test_ts, nbeats_pred)

print(f"N-BEATS Test MAE: ${nbeats_mae:,.0f}")
print(f"N-BEATS Test RMSE: ${nbeats_rmse:,.0f}")

In [ ]:
nbeats_forecast = nbeats.predict(HORIZON)
nbeats_dec_2026 = nbeats_forecast.values()[-1][0]

print(f"N-BEATS Dec 2026: ${nbeats_dec_2026/1e6:.2f}M")

In [ ]:
from darts.models import ChronosModel

chronos = ChronosModel()
chronos.fit(train_ts)
chronos_forecast = chronos.predict(HORIZON)
chronos_dec_2026 = chronos_forecast.values()[-1][0]

print(f"Chronos-2 Dec 2026: ${chronos_dec_2026/1e6:.2f}M")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(revenue.index, revenue / 1e6, 'k-', label='Actual')

nbeats_index = pd.date_range('2026-01-01', periods=HORIZON, freq='MS')
ax.plot(nbeats_index, nbeats_forecast.values() / 1e6, 'g--', label='N-BEATS')
ax.plot(nbeats_index, chronos_forecast.values() / 1e6, 'm--', label='Chronos-2')

ax.axhline(y=VP_TARGET / 1e6, color='r', linestyle=':', label='VP Target: $4.1M')

ax.set_ylabel('Revenue ($M)')
ax.set_title('Phase 3: Modern Model Forecasts')
ax.legend()
plt.tight_layout()
plt.show()

---

## Phase 4: Comparison and Recommendation

In [ ]:
comparison_data = {
    'Model': ['Naive', '6MA', '12MA', 'Trend', 'ARIMA', 'SARIMAX', 'N-BEATS', 'Chronos-2'],
    'Dec 2026 Forecast': [
        naive_fc.iloc[-1],
        ma6_fc.iloc[-1],
        ma12_fc.iloc[-1],
        trend_fc.iloc[-1],
        arima_dec_2026,
        sarimax_dec_2026,
        nbeats_dec_2026,
        chronos_dec_2026
    ],
    '95% CI Lower': [
        np.nan, np.nan, np.nan, np.nan,
        arima_lower,
        sarimax_lower,
        np.nan, np.nan
    ],
    '95% CI Upper': [
        np.nan, np.nan, np.nan, np.nan,
        arima_upper,
        sarimax_upper,
        np.nan, np.nan
    ],
    'Test MAE': [
        np.nan, np.nan, np.nan, np.nan,
        arima_mae,
        sarimax_mae,
        nbeats_mae,
        np.nan
    ],
    'P(Reach $4.1M)': [
        np.nan, np.nan, np.nan, np.nan,
        arima_prob,
        sarimax_prob,
        np.nan, np.nan
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df['Dec 2026 Forecast'] = comparison_df['Dec 2026 Forecast'] / 1e6
comparison_df['95% CI Lower'] = comparison_df['95% CI Lower'] / 1e6
comparison_df['95% CI Upper'] = comparison_df['95% CI Upper'] / 1e6

print(comparison_df.to_string(index=False))

### Written Recommendation

Write a 500–1000 word stakeholder-ready recommendation addressing:

1. **What is the realistic range for December 2026 revenue?**
   - Based on model forecasts and prediction intervals
   - Consider the spread across different approaches

2. **What should the company plan for instead of $4.1M?**
   - Provide a specific target or range
   - Explain the gap between models and the VP's original target
   - Discuss probability of achieving different revenue levels

3. **Which forecasting approach would you recommend for ongoing use, and why?**
   - Compare accuracy, interpretability, and practical considerations
   - Consider maintenance requirements and stakeholder understanding
   - Recommend a specific approach with justification

**YOUR WRITTEN RECOMMENDATION (500–1000 words):**

[Write your recommendation here...]